# Semantic Parser Audit — Gemini

For every model result row, Gemini looks at `Raw_Output` and `Reasoning` (where present) and extracts the final answer the model reached — even if the regex parser missed it.

Outputs an audited CSV with `audited_answer` and `audited_correct` columns added.

In [ ]:
df = pd.read_excel("/content/puzzle_answers_cot_gpt40.xlsx")

In [ ]:
df.to_csv("puzzle_answers_cot_gpt40.csv",index=False)

In [ ]:
df = pd.read_excel("/content/puzzle_answers_zero_shot_gpt40.xlsx")

In [ ]:
df = pd.read_excel("/content/puzzle_answers_zero_shot_o3.xlsx")

In [ ]:
df.to_csv("puzzle_answers_zero_shot_o3.csv",index=False)

## 1. Install & Imports

In [ ]:
!pip install -q google-genai pandas openpyxl tqdm

In [ ]:
import re
import os
import json
import time
import glob
import pandas as pd
from tqdm import tqdm
from google import genai
from google.genai import types

## 2. Config

In [ ]:
# ── API ───────────────────────────────────────────────────────────────────────
GEMINI_API_KEY = 'REDACTED_USE_ENVIRONMENT_VARIABLE'
MODEL_ID       = 'gemini-2.5-flash'

# ── Paths ─────────────────────────────────────────────────────────────────────
# Ground truth
GT_PATH  = '/content/puzzle_samples.xlsx'   # has Puzzle_Number, Question_Number, Answer, Solved_Percentile

# Input result files — one per model/run
# Keys are display names, values are file paths
RESULT_FILES = {
    #'claude-haiku-cot':    '/content/puzzle_cot_claude_haiku45_original.csv',
    #'deepseek-r1-7b':        '/content/puzzle_cot_deepseek_r1_qwen7b.csv',
    'gemini-2.5-flash':      '/content/puzzle_cot_gemini25flash.csv',
    #'gpt-4o-cot':          '/content/puzzle_answers_cot_gpt40.csv',
    #'llama3.1:8b-cot':           '/content/puzzle_cot_llama3.1_8b.csv',
    #'o3':               '/content/puzzle_answers_zero_shot_o3.csv',
    #'claude-haiku-zs':     '/content/puzzle_zeroshot_claude_haiku45.csv',
    #'gpt-4o-zs':           '/content/puzzle_answers_zero_shot_gpt40.csv',
    #'llama3.1:8b-zs':            '/content/puzzle_zeroshot_llama3.1_8b.csv',
    #'qwen3.5:9b':             '/content/puzzle_zeroshot_qwen3.5_9b.csv',  # partially visible
}

WORKDIR   = './audit_outputs'
os.makedirs(WORKDIR, exist_ok=True)

SAVE_EVERY        = 20
MAX_OUTPUT_TOKENS = 256    # answer is just one letter — keep it tiny
THINKING_BUDGET   = 0      # no thinking needed

# Run on ALL rows (not just missing answers) for full coverage
# Set to True to only audit rows where Model_Answer is missing/invalid
ONLY_MISSING = True

VALID_ANSWERS = {'A', 'B', 'C', 'D', 'E'}

## 3. Init Client

In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)
print(f'Client ready. Model: {MODEL_ID}')

## 4. Load Ground Truth

In [ ]:
if GT_PATH.endswith('.xlsx'):
    gt = pd.read_excel(GT_PATH)
else:
    gt = pd.read_csv(GT_PATH)

gt = gt[['Puzzle_Number', 'Question_Number', 'Answer', 'Solved_Percentile']].copy()
gt.columns = ['Puzzle_Number', 'Question_Number', 'GT_Answer', 'Solved_Percentile']

def map_difficulty(p):
    if p < 30:   return 'Hard'
    elif p < 60: return 'Medium'
    else:        return 'Easy'

gt['difficulty'] = gt['Solved_Percentile'].apply(map_difficulty)
print(f'Ground truth rows: {len(gt)}')
print(gt['difficulty'].value_counts().to_string())

## 5. Helpers

In [ ]:
def load_completed(path, key_cols):
    if not os.path.exists(path):
        return [], set()
    done_df   = pd.read_csv(path)
    done_keys = set(zip(*[done_df[c] for c in key_cols]))
    records   = done_df.to_dict('records')
    print(f'  Resuming: {len(records)} rows already done.')
    return records, done_keys


def generate_audit(system_prompt, user_prompt):
    max_retries = 5
    wait = 30
    for attempt in range(max_retries):
        try:
            t0 = time.time()
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET),
                ),
            )
            elapsed = (time.time() - t0) / 60.0
            raw = ''
            for part in response.candidates[0].content.parts:
                if not getattr(part, 'thought', False):
                    raw += (part.text or '')
            return raw.strip(), elapsed
        except Exception as e:
            if attempt < max_retries - 1 and ('503' in str(e) or '429' in str(e)):
                print(f'  [Retry {attempt+1}/{max_retries}] {e} — waiting {wait}s')
                time.sleep(wait)
                wait *= 2
            else:
                raise


def parse_answer(raw: str) -> str:
    """Extract A/B/C/D/E from Gemini's response."""
    raw = raw.strip().upper()
    # direct single letter
    if raw in VALID_ANSWERS:
        return raw
    # first valid letter found
    m = re.search(r'\b([ABCDE])\b', raw)
    if m:
        return m.group(1)
    return 'UNKNOWN'


def needs_audit(row) -> bool:
    """Return True if this row should be sent to Gemini."""
    if ONLY_MISSING:
        answer = str(row.get('Model_Answer', '') or '').strip().upper()
        return answer not in VALID_ANSWERS
    return True

## 6. Audit System Prompt

In [ ]:
AUDIT_SYSTEM_PROMPT = """\
You are extracting the final answer from a model's response to a multiple-choice logic puzzle.
The answer must be one of: A, B, C, D, or E.

Instructions:
- Look through the entire text provided (raw output and reasoning if present).
- Find the model's final committed answer choice.
- If the model changed its mind, use the LAST answer it committed to.
- If the model clearly reached an answer, output just that single letter.
- If the model did not reach any answer or only expressed uncertainty without committing, output: NONE

Output ONLY a single letter (A/B/C/D/E) or the word NONE. Nothing else.
"""

## 7. Run Audit

In [ ]:
def run_audit_for_model(model_name: str, filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath)
    out_path = os.path.join(WORKDIR, f'audit_{model_name}.csv')
    KEY_COLS  = ['Puzzle_Number', 'Question_Number','Sample']
    records, done = load_completed(out_path, KEY_COLS)

    print(f'\n--- {model_name} ({len(df)} rows) ---')

    for _, row in tqdm(df.iterrows(), total=len(df), desc=model_name):
        key = (row['Puzzle_Number'], row['Question_Number'],row['Sample'])
        if key in done:
            continue

        # Build user prompt from whatever columns exist
        raw_output = str(row.get('Raw_Output', '') or '')
        reasoning  = str(row.get('Reasoning',  '') or '')
        existing   = str(row.get('Model_Answer', '') or '').strip().upper()

        user_prompt = f'Raw output:\n{raw_output}'
        if reasoning and reasoning.lower() not in ('nan', '', 'none'):
            user_prompt += f'\n\nReasoning:\n{reasoning}'
        if existing in VALID_ANSWERS:
            user_prompt += f'\n\nPreviously parsed answer: {existing}'

        audited_raw, elapsed = '', 0.0
        audited_answer = 'UNKNOWN'

        if needs_audit(row):
            try:
                audited_raw, elapsed = generate_audit(AUDIT_SYSTEM_PROMPT, user_prompt)
                audited_answer = parse_answer(audited_raw)
            except Exception as e:
                print(f'  [P{row["Puzzle_Number"]} Q{row["Question_Number"]}] Error: {e}')
                audited_answer = 'ERROR'
        else:
            audited_answer = existing   # already valid, keep it

        records.append({
            **row.to_dict(),
            'model_name':      model_name,
            'audited_answer':  audited_answer,
            'audited_raw':     audited_raw,
            'audit_elapsed':   round(elapsed, 3),
        })
        done.add(key)

        if len(records) % SAVE_EVERY == 0:
            pd.DataFrame(records).to_csv(out_path, index=False)
            print(f'  [Checkpoint] {len(records)} rows')

    pd.DataFrame(records).to_csv(out_path, index=False)
    print(f'  Done: {len(records)} rows -> {out_path}')
    return pd.DataFrame(records)


# Run for all models
all_audited = {}
for model_name, filepath in RESULT_FILES.items():
    if not os.path.exists(filepath):
        print(f'  SKIPPING {model_name} — file not found: {filepath}')
        continue
    all_audited[model_name] = run_audit_for_model(model_name, filepath)

## 8. Merge Ground Truth + Mark Correct

In [ ]:
combined_frames = []
for model_name, df in all_audited.items():
    merged = df.merge(gt, on=['Puzzle_Number', 'Question_Number'], how='left')
    merged['correct'] = merged['audited_answer'] == merged['GT_Answer']
    combined_frames.append(merged)

combined = pd.concat(combined_frames, ignore_index=True)
combined.to_csv(os.path.join(WORKDIR, 'all_models_audited.csv'), index=False)

print(f'Combined audited rows: {len(combined)}')
print('\nAnswer recovery summary:')
for model_name, g in combined.groupby('model_name'):
    n          = len(g)
    unknown    = (g['audited_answer'] == 'UNKNOWN').sum()
    no_answer  = (g['audited_answer'] == 'NONE').sum()
    recovered  = n - unknown - no_answer
    print(f'  {model_name:<25} total={n}  recovered={recovered}  unknown={unknown}  no_answer={no_answer}')